In [ ]:
repository_filter: list[str] = []
health_metric: str = "maintainabilityIndex"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.express as px
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/class_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
df = df.dropna(subset=["className", "lineCount"])
df = df[df["lineCount"] > 0]
df["package"] = df["package"].fillna("(default)")

# Limit to top 15 repos by class count
repo_counts = df.groupby("repoShort").size().nlargest(15)
df = df[df["repoShort"].isin(repo_counts.index)]

# Shorten deep package names
df["package"] = df["package"].apply(
    lambda p: ".".join(str(p).split(".")[-2:]) if len(str(p).split(".")) > 2 else str(p)
)

if len(df) == 0:
    fig = qu.empty_figure()
else:
    fig = px.sunburst(
        df,
        path=["repoShort", "package", "classShort"],
        values="lineCount",
        color="maintainabilityIndex",
        color_continuous_scale=["#EF5350", "#FF8A65", "#FFE082", "#A5D6A7", "#4CAF50"],
        color_continuous_midpoint=50,
    )
    n = df["repoShort"].nunique()
    fig.update_layout(
        title=f"Portfolio Code Health ({n} largest repos)",
        height=800,
        width=900,
        margin=dict(t=50, l=0, r=0, b=0),
    )
fig.show()